In [47]:
import laspy
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import ConvexHull
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage
import time

# ==========================================
#   CONFIGURATION
# ==========================================
INPUT_FILE = Path("demo.laz")
OUTPUT_DIR = Path("Auto_Extracted_Results")
REPORT_FILE = OUTPUT_DIR / "Auto_Inventory.csv"

GRID_SIZE = 0.1             # CHM resolution
MIN_TREE_HEIGHT = 2         # Minimum canopy height (meters)
MIN_DISTANCE_PIXELS = 7     # Tree-top separation (pixels)

# ==========================================
#   FUNCTIONS
# ==========================================

def load_laz(path):
    """Load .laz file using laspy."""
    if not path.exists():
        print(f"[ERROR] File not found: {path}")
        return None, None
    
    las = laspy.read(str(path))
    xyz = np.vstack((las.x, las.y, las.z)).T
    return las, xyz


def create_chm(xyz, grid_size):
    """
    Creates a 2D Canopy Height Model (CHM) from point cloud.
    RETURNS: chm, x_min, y_min, AND ground_level (Fix applied here)
    """
    x_min, x_max = xyz[:, 0].min(), xyz[:, 0].max()
    y_min, y_max = xyz[:, 1].min(), xyz[:, 1].max()

    cols = int((x_max - x_min) / grid_size) + 1
    rows = int((y_max - y_min) / grid_size) + 1

    print(f"   -> Creating CHM Grid: {rows}x{cols} pixels")

    chm = np.zeros((rows, cols))

    # Grid indexing
    x_idx = ((xyz[:, 0] - x_min) / grid_size).astype(int)
    y_idx = ((xyz[:, 1] - y_min) / grid_size).astype(int)

    df = pd.DataFrame({'r': y_idx, 'c': x_idx, 'z': xyz[:, 2]})
    grid_max = df.groupby(['r', 'c'])['z'].max().reset_index()

    chm[grid_max['r'], grid_max['c']] = grid_max['z']

    # Normalize by removing ground
    # NOTE: Using 1st percentile as Global Ground approximation for Demo
    ground_level = np.percentile(xyz[:, 2], 1)
    
    chm_normalized = chm - ground_level
    chm_normalized[chm_normalized < 0] = 0

    return chm_normalized, x_min, y_min, ground_level  # <--- Returning Ground Level


def calculate_features(x, y, z, ground_z):
    """Calculates Height, Crown Diameter, Area of tree."""
    
    # --- HEIGHT FIX ---
    # Old Logic: max(z) - min(z) -> Wrong (Measures Crown Depth)
    # New Logic: max(z) - ground_z -> Correct (Measures Tree Height)
    height = np.max(z) - ground_z
    
    width_x = np.max(x) - np.min(x)
    width_y = np.max(y) - np.min(y)
    diameter = (width_x + width_y) / 2

    try:
        hull = ConvexHull(np.vstack([x, y]).T)
        area = hull.volume  # 2D hull area
    except:
        area = 0.0

    return height, diameter, area


# ==========================================
#   MAIN PROGRAM
# ==========================================

if __name__ == "__main__":
    print("### STARTING AUTOMATED (UNSUPERVISED) EXTRACTION ###")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # -------------------------------
    # Load point cloud
    # -------------------------------
    print(f"Loading {INPUT_FILE}...")
    las, xyz = load_laz(INPUT_FILE)
    if las is None:
        exit()

    # -------------------------------
    # Create CHM
    # -------------------------------
    print("Generating Canopy Height Model...")
    # FIX: Catching ground_level here
    chm, x_off, y_off, global_ground = create_chm(xyz, GRID_SIZE)

    # Smooth the CHM
    chm = ndimage.gaussian_filter(chm, sigma=1)

    # -------------------------------
    # Tree-top detection
    # -------------------------------
    print("Detecting Tree Tops...")
    local_maxi = peak_local_max(
        chm,
        min_distance=MIN_DISTANCE_PIXELS,
        threshold_abs=MIN_TREE_HEIGHT
    )
    print(f"   -> Detected {len(local_maxi)} potential trees.")

    # Assign markers for watershed
    markers = np.zeros_like(chm, dtype=int)
    for i, (r, c) in enumerate(local_maxi):
        markers[r, c] = i + 1

    # -------------------------------
    # Segmentation
    # -------------------------------
    print("Running Watershed Segmentation...")
    labels = watershed(-chm, markers, mask=chm > MIN_TREE_HEIGHT)

    # -------------------------------
    # Extract trees
    # -------------------------------
    print("Extracting Point Clouds (This is the heavy part)...")

    x_idx = ((xyz[:, 0] - x_off) / GRID_SIZE).astype(int)
    y_idx = ((xyz[:, 1] - y_off) / GRID_SIZE).astype(int)

    max_r, max_c = labels.shape
    valid_mask = (x_idx >= 0) & (x_idx < max_c) & (y_idx >= 0) & (y_idx < max_r)

    point_tree_ids = np.zeros(len(xyz), dtype=int)
    point_tree_ids[valid_mask] = labels[y_idx[valid_mask], x_idx[valid_mask]]

    unique_ids = np.unique(point_tree_ids)
    inventory = []

    # -------------------------------
    # Save each extracted tree
    # -------------------------------
    for tree_id in unique_ids:
        if tree_id == 0:
            continue

        mask = point_tree_ids == tree_id

        if np.count_nonzero(mask) > 50:
            sub_las = laspy.LasData(las.header)
            sub_las.points = las.points[mask]

            # Calculate features
            # FIX: Passing global_ground to calculation
            h, cd, area = calculate_features(sub_las.x, sub_las.y, sub_las.z, global_ground)

            # Save point cloud
            fname = f"Auto_Tree_{tree_id}.laz"
            sub_las.write(str(OUTPUT_DIR / fname))

            # Store inventory record
            inventory.append({
                "Tree_ID": f"Tree_{tree_id}",
                "X_Location": round(np.mean(sub_las.x), 2),
                "Y_Location": round(np.mean(sub_las.y), 2),
                "Height_m": round(h, 2),
                "Diameter_m": round(cd, 2),
                "Points": np.count_nonzero(mask)
            })

    # -------------------------------
    # Save inventory CSV
    # -------------------------------
    if inventory:
        df = pd.DataFrame(inventory)
        df.to_csv(REPORT_FILE, index=False)

        print("\n" + "=" * 40)
        print("        AUTOMATION RESULTS")
        print("=" * 40)
        print(f"Total Trees Extracted: {len(df)}")
        print(f"Files saved in: {OUTPUT_DIR}")
        print(f"Report saved as: {REPORT_FILE}")
    else:
        print("No trees found.")

### STARTING AUTOMATED (UNSUPERVISED) EXTRACTION ###
Loading demo.laz...
Generating Canopy Height Model...
   -> Creating CHM Grid: 237x271 pixels
Detecting Tree Tops...
   -> Detected 49 potential trees.
Running Watershed Segmentation...
Extracting Point Clouds (This is the heavy part)...

        AUTOMATION RESULTS
Total Trees Extracted: 49
Files saved in: Auto_Extracted_Results
Report saved as: Auto_Extracted_Results\Auto_Inventory.csv


In [49]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ===================================================
#   LOAD FILES
# ===================================================
detected_file = "Auto_Extracted_Results/Auto_Inventory.csv"
ground_truth_file = "Demo_Ground_Truth.csv"

det = pd.read_csv(detected_file)
gt = pd.read_csv(ground_truth_file)

print("Loaded:")
print(f"  Detected Trees   = {len(det)}")
print(f"  Ground Truth     = {len(gt)}")


# ===================================================
#   BUILD KD-TREE FOR NEAREST MATCHING (XY)
# ===================================================
gt_coords = np.vstack([
    gt["Easting [m]"], 
    gt["Northing [m]"]
]).T

det_coords = np.vstack([
    det["X_Location"], 
    det["Y_Location"]
]).T

tree = cKDTree(gt_coords)

distances, indices = tree.query(det_coords)

det["Matched_GT_Index"] = indices
det["XY_Error_m"] = distances


# ===================================================
#   MERGE MATCHED DATA
# ===================================================
merged = det.join(
    gt.add_prefix("GT_").iloc[indices].reset_index(drop=True)
)

# ===================================================
#   CALCULATE ERRORS
# ===================================================
merged["Height_Error"] = merged["Height_m"] - merged["GT_Height [m]"]
merged["CrownDiameter_Error"] = merged["Diameter_m"] - merged["GT_Crown diameter [m]"]

merged["Height_Abs_Error"] = merged["Height_Error"].abs()
merged["CrownDiameter_Abs_Error"] = merged["CrownDiameter_Error"].abs()


# ===================================================
#   ACCURACY METRICS
# ===================================================
accuracy_report = {
    "Total_Detected": len(det),
    "Total_GroundTruth": len(gt),
    "Matched_Trees": len(merged),
    "Match_Percentage_%": round(len(merged) / len(gt) * 100, 2),

    "XY_RMSE_m": round(np.sqrt(np.mean(merged["XY_Error_m"] ** 2)), 3),
    "XY_Mean_Error_m": round(merged["XY_Error_m"].mean(), 3),

    "Height_MAE_m": round(merged["Height_Abs_Error"].mean(), 3),
    "Height_RMSE_m": round(np.sqrt(np.mean(merged["Height_Error"] ** 2)), 3),

    "CD_MAE_m": round(merged["CrownDiameter_Abs_Error"].mean(), 3),
    "CD_RMSE_m": round(
        np.sqrt(np.mean(merged["CrownDiameter_Error"] ** 2)), 3),
}

# Print results
print("\n===================== ACCURACY REPORT =====================")
for k, v in accuracy_report.items():
    print(f"{k:25s} : {v}")

# Save results
merged.to_csv("Accuracy_Matched_Trees.csv", index=False)
pd.DataFrame([accuracy_report]).to_csv("Accuracy_Report.csv", index=False)

print("\nFull accuracy table saved as: Accuracy_Matched_Trees.csv")
print("Summary report saved as: Accuracy_Report.csv")


Loaded:
  Detected Trees   = 49
  Ground Truth     = 51

===================== ACCURACY REPORT =====================
Total_Detected            : 49
Total_GroundTruth         : 51
Matched_Trees             : 49
Match_Percentage_%        : 96.08
XY_RMSE_m                 : 1.988
XY_Mean_Error_m           : 1.69
Height_MAE_m              : 7.432
Height_RMSE_m             : 10.054
CD_MAE_m                  : 3.87
CD_RMSE_m                 : 4.934

Full accuracy table saved as: Accuracy_Matched_Trees.csv
Summary report saved as: Accuracy_Report.csv
